# Reinforcement Learning: CartPole

In the scikit-learn notebooks, every training example came with the right answer attached. **Reinforcement learning** is different: nobody tells the agent the correct move. It tries things, sees what happens, and gets a **reward** — a single number saying "that went well" or "that went badly". Over thousands of attempts it works out a strategy on its own.

The loop is always the same:

1. The agent **observes** the situation.
2. It picks an **action**.
3. The environment returns a **reward** and the next situation.
4. The agent **learns** a little from what just happened.

This notebook trains an agent with **Q-learning** — no neural network, just a table of numbers it fills in through trial and error.

# The CartPole Environment

CartPole is the classic first problem in reinforcement learning. A pole is balanced on a cart. Each step, the agent may push the cart **left** or **right**.

- **Reward:** `+1` for every step the pole stays up.
- **The attempt ends** if the pole tips more than 12 degrees, the cart rolls off the screen, or 500 steps pass.

So "get as much reward as possible" means exactly "keep the pole balanced as long as possible".

The environment comes from **Gymnasium**, the standard toolkit of reinforcement-learning practice problems.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import gymnasium as gym
import numpy as np

# render_mode="human" draws the cart and pole in the cell as it runs.
env = gym.make("CartPole-v1", render_mode="human", disable_env_checker=True)

# Three attempts. The agent pushes left or right at random, with no plan.
for attempt in range(3):
  obs, info = env.reset()
  done = False
  while not done:
    obs, reward, terminated, truncated, info = env.step(env.action_space.sample())
    done = terminated or truncated
  # The pole is past the point of recovery - keep going with no correction
  # so you can watch it topple right over before the next attempt.
  for _ in range(20):
    obs, _, _, _, _ = env.step(0 if obs[2] > 0 else 1)

env.close()
print("A random agent has no plan - every attempt ends with the pole falling over.")

# What the Agent Sees

Every step, the environment hands back an **observation**: four numbers describing the current situation.

| Number | Meaning |
|---|---|
| `obs[0]` | cart position (how far left or right) |
| `obs[1]` | cart velocity (how fast it is sliding) |
| `obs[2]` | pole angle (how far it is leaning) |
| `obs[3]` | pole angular velocity (how fast it is tipping) |

The pole angle and how fast it is tipping matter most - those are what warn you the pole is about to fall.

In [ ]:
env = gym.make("CartPole-v1")
obs, info = env.reset(seed=0)

print("cart position        :", round(float(obs[0]), 3))
print("cart velocity        :", round(float(obs[1]), 3))
print("pole angle           :", round(float(obs[2]), 3))
print("pole angular velocity:", round(float(obs[3]), 3))

# From Numbers to Buckets

Q-learning keeps a table with one row per situation. But those four numbers are decimals - there are infinitely many possible situations, so we cannot have a row for each.

The fix: chop each number into a few **buckets**. Instead of "pole angle = 0.0413" the agent just knows "pole angle is in bucket 7 of 12". We give the two angle-related numbers more buckets, because small changes there matter more.

`4 numbers` times `a handful of buckets each` = a table the agent can actually fill in.

In [ ]:
# How many buckets for each of the 4 observation values.
BINS = [6, 6, 12, 12]

# The range we expect each value to fall in (angle and velocities are clipped).
LOW  = np.array([-2.4, -3.0, -0.21, -3.5])
HIGH = np.array([ 2.4,  3.0,  0.21,  3.5])

# The cut points between buckets for each value.
edges = [np.linspace(LOW[i], HIGH[i], BINS[i] + 1)[1:-1] for i in range(4)]

def to_state(obs):
  """Turn 4 decimals into a tuple of 4 bucket numbers."""
  return tuple(int(np.digitize(obs[i], edges[i])) for i in range(4))

print("raw observation:", np.round(obs, 3))
print("bucketed state :", to_state(obs))

# The Q-Table

The **Q-table** stores, for every bucketed situation, one number per action:

> *How much total reward do I expect if I take this action here, and then keep playing well?*

It starts as all zeros - the agent knows nothing. Training slowly fills it in. To choose a move, the agent looks up the current situation and picks the action with the bigger number.

In [ ]:
n_actions = env.action_space.n   # 2: push left, push right
q_table = np.zeros(BINS + [n_actions])   # shape (6, 6, 12, 12, 2)

print("Q-table shape       :", q_table.shape)
print("situations x actions :", q_table.size)
print("all starts at zero   :", q_table.sum() == 0)

# The Learning Rule

**Explore vs. exploit.** Early on the agent should try random moves to discover what works. Later it should trust what it has learned. `epsilon` is the chance of a random move; we start it near `1.0` and let it shrink.

**The update.** After taking an action and seeing the reward, nudge that table entry toward a better estimate:

```
Q[state, action]  +=  lr * ( reward  +  gamma * max(Q[next_state])  -  Q[state, action] )
```

- `reward` - what we just got (`+1`).
- `gamma * max(Q[next_state])` - how good the *next* situation looks. `gamma` (0 to 1) sets how much we care about the future.
- `lr` - the learning rate: how big a nudge each step gives.

Repeat this a few million times and the table becomes a genuinely good strategy.

In [ ]:
EPISODES = 6000  #@param {type:"slider", min:1000, max:15000, step:1000}
LEARNING_RATE = 0.1
DISCOUNT = 0.99
MIN_EPSILON = 0.05

# no render_mode so training runs fast; disable_env_checker skips per-step validation
train_env = gym.make("CartPole-v1", disable_env_checker=True)
rewards = []

try:
  for episode in range(EPISODES):
    obs, _ = train_env.reset()
    state = to_state(obs)
    # epsilon shrinks from 1.0 to MIN_EPSILON over the first 70% of training
    epsilon = max(MIN_EPSILON, 1.0 - episode / (EPISODES * 0.7))
    done = False
    total = 0.0

    while not done:
      if np.random.random() < epsilon:
        action = train_env.action_space.sample()        # explore
      else:
        action = int(np.argmax(q_table[state]))          # exploit

      obs, reward, terminated, truncated, _ = train_env.step(action)
      done = terminated or truncated
      next_state = to_state(obs)
      total += reward

      # bootstrap from the next state, unless the pole actually fell
      best_next = np.max(q_table[next_state]) * (not terminated)
      td_target = reward + DISCOUNT * best_next
      q_table[state + (action,)] += LEARNING_RATE * (td_target - q_table[state + (action,)])
      state = next_state

    rewards.append(total)
    if (episode + 1) % 500 == 0:
      recent = np.mean(rewards[-500:])
      print(f"episode {episode + 1:5d}   epsilon {epsilon:.2f}   avg reward (last 500): {recent:6.1f}")
except KeyboardInterrupt:
  print(f"\nStopped early at episode {len(rewards)}.")

train_env.close()
if rewards:
  print(f"\nDone. Best single attempt: {max(rewards):.0f} steps balanced.")

# Did It Learn?

Plot the reward for every attempt, plus a running average. A line that climbs from around 20 up toward the hundreds means the agent is genuinely getting better - not memorising, but discovering a strategy from reward alone.

In [ ]:
import matplotlib.pyplot as plt

rewards_arr = np.array(rewards)
window = 100
moving_avg = np.convolve(rewards_arr, np.ones(window) / window, mode="valid")

plt.figure(figsize=(9, 4))
plt.plot(rewards_arr, alpha=0.2, label="each attempt")
plt.plot(range(window - 1, len(rewards_arr)), moving_avg, linewidth=2, label=f"{window}-attempt average")
plt.xlabel("attempt")
plt.ylabel("steps the pole stayed up")
plt.title("Learning progress")
plt.legend()
plt.show()

# Watch the Trained Agent

Now run the agent with **no random moves** - always its best-known action - and draw it. Compare this with the random agent from the start of the notebook.

In [ ]:
import gymnasium as gym   # this import also sets up the in-browser drawing

show_env = gym.make("CartPole-v1", render_mode="human", disable_env_checker=True)

for attempt in range(3):
  obs, _ = show_env.reset()
  state = to_state(obs)
  done = False
  steps = 0
  while not done:
    action = int(np.argmax(q_table[state]))
    obs, reward, terminated, truncated, _ = show_env.step(action)
    state = to_state(obs)
    done = terminated or truncated
    steps += 1
  print(f"attempt {attempt + 1}: balanced for {steps} steps")

show_env.close()

# Things to Try

- Drag `EPISODES` higher and retrain (re-run the Q-table cell first to start fresh). How far does the average climb?
- Change `LEARNING_RATE` to `0.5` or `0.01`. Faster is not always better.
- Set `MIN_EPSILON = 0.0`. What happens when the agent never explores near the end?
- Give the pole angle fewer buckets (`BINS = [6, 6, 4, 4]`) and retrain. Why does coarser bucketing hurt?

# Check Your Understanding